# C++ DSA Practice with Jupyter

This notebook demonstrates how to run C++ code directly from Jupyter cells by compiling and executing it through Python.

## Concept: Wrapper Approach

Instead of using xeus-cling (which is difficult to install on macOS), we use Python to:
1. Write C++ code to a temporary file
2. Compile it using g++
3. Execute the binary
4. Capture and display output

This gives you the best of both worlds - interactive Python + compiled C++ performance.

## Setup: Install Helper Functions

In [1]:
import subprocess
import tempfile
import os
from pathlib import Path

def run_cpp(code, compiler_flags="-std=c++17 -Wall -Wextra -O2"):
    """
    Compile and run C++ code from a Jupyter cell.
    
    Parameters:
    -----------
    code : str
        The C++ code to execute
    compiler_flags : str
        Compilation flags (default: C++17 with optimizations)
    
    Returns:
    --------
    output : str
        The stdout from the compiled program
    
    Example:
    --------
    >>> cpp_code = '''
    ... #include <iostream>
    ... int main() {
    ...     std::cout << "Hello from C++" << std::endl;
    ...     return 0;
    ... }
    ... '''
    >>> run_cpp(cpp_code)
    """
    
    with tempfile.TemporaryDirectory() as tmpdir:
        # Create source and output file paths
        src_file = os.path.join(tmpdir, "program.cpp")
        bin_file = os.path.join(tmpdir, "program")
        
        # Write C++ code to file
        with open(src_file, 'w') as f:
            f.write(code)
        
        # Compile the code
        compile_cmd = f"g++ {compiler_flags} {src_file} -o {bin_file}"
        try:
            result = subprocess.run(
                compile_cmd,
                shell=True,
                capture_output=True,
                text=True,
                timeout=10
            )
            
            if result.returncode != 0:
                print("Compilation Error:")
                print(result.stderr)
                return None
            
            # Run the compiled program
            exec_result = subprocess.run(
                [bin_file],
                capture_output=True,
                text=True,
                timeout=10
            )
            
            if exec_result.returncode != 0:
                print("Runtime Error:")
                print(exec_result.stderr)
                return None
            
            return exec_result.stdout
        
        except subprocess.TimeoutExpired:
            print("Error: Program execution timed out")
            return None
        except Exception as e:
            print(f"Error: {e}")
            return None

print("Helper function 'run_cpp()' is now available!")
print("You can now write C++ code in the cells below.")

Helper function 'run_cpp()' is now available!
You can now write C++ code in the cells below.


## Example 1: Basic Hello World

In [2]:
cpp_code = """
#include <iostream>

int main() {
    std::cout << "Hello from C++!" << std::endl;
    std::cout << "Running C++ code in Jupyter notebook" << std::endl;
    return 0;
}
"""

output = run_cpp(cpp_code)
if output:
    print("Output:")
    print(output)

Output:
Hello from C++!
Running C++ code in Jupyter notebook



## Example 2: Binary Search Algorithm

In [3]:
cpp_code = """
#include <iostream>
#include <vector>

int binarySearch(const std::vector<int>& arr, int target) {
    int left = 0, right = arr.size() - 1;
    
    while (left <= right) {
        int mid = left + (right - left) / 2;
        
        if (arr[mid] == target) {
            return mid;  // Found
        } else if (arr[mid] < target) {
            left = mid + 1;  // Search right half
        } else {
            right = mid - 1;  // Search left half
        }
    }
    
    return -1;  // Not found
}

int main() {
    std::vector<int> arr = {2, 5, 8, 12, 16, 23, 38, 45, 56, 67};
    int target = 23;
    
    int result = binarySearch(arr, target);
    
    if (result != -1) {
        std::cout << "Element found at index: " << result << std::endl;
    } else {
        std::cout << "Element not found" << std::endl;
    }
    
    return 0;
}
"""

output = run_cpp(cpp_code)
if output:
    print("Binary Search Output:")
    print(output)

Binary Search Output:
Element found at index: 5



## Example 3: Recursion - Fibonacci

In [ ]:
cpp_code = """
#include <iostream>

// Recursive Fibonacci with poor performance (exponential time)
int fib_naive(int n) {
    if (n <= 1) return n;
    return fib_naive(n - 1) + fib_naive(n - 2);
}

// Optimized Fibonacci with memoization
int fib_memo(int n, int memo[]) {
    if (n <= 1) return n;
    
    if (memo[n] != -1) {
        return memo[n];
    }
    
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo);
    return memo[n];
}

int main() {
    int n = 35;
    
    // Using naive recursion
    std::cout << "Fibonacci (naive) of " << n << ": " << fib_naive(n) << std::endl;
    
    // Using memoization
    int memo[100];
    for (int i = 0; i < 100; i++) memo[i] = -1;
    std::cout << "Fibonacci (memo) of " << n << ": " << fib_memo(n, memo) << std::endl;
    
    return 0;
}
"""

output = run_cpp(cpp_code)
if output:
    print("Fibonacci Output:")
    print(output)

## Example 4: Dynamic Programming - Coin Change

In [ ]:
cpp_code = """
#include <iostream>
#include <vector>
#include <algorithm>

int coinChange(std::vector<int>& coins, int amount) {
    // dp[i] = minimum number of coins to make amount i
    std::vector<int> dp(amount + 1, INT_MAX);
    dp[0] = 0;  // Base case: 0 coins needed to make 0
    
    for (int i = 1; i <= amount; i++) {
        for (int coin : coins) {
            if (coin <= i && dp[i - coin] != INT_MAX) {
                dp[i] = std::min(dp[i], dp[i - coin] + 1);
            }
        }
    }
    
    return (dp[amount] == INT_MAX) ? -1 : dp[amount];
}

int main() {
    std::vector<int> coins = {1, 2, 5};
    int amount = 5;
    
    int result = coinChange(coins, amount);
    
    std::cout << "Minimum coins needed to make amount " << amount << ": "
              << result << std::endl;
    
    return 0;
}
"""

output = run_cpp(cpp_code)
if output:
    print("Coin Change Output:")
    print(output)

## How to Use This Notebook

1. **Write C++ Code**: Create a new code cell below
2. **Define your cpp_code variable** with your C++ code
3. **Call run_cpp(cpp_code)** to compile and execute
4. **View the output** immediately

### Template for New C++ Cells:

```python
cpp_code = """
#include <iostream>
#include <vector>

int main() {
    // Your C++ code here
    std::cout << "Result: " << result << std::endl;
    return 0;
}
"""

output = run_cpp(cpp_code)
if output:
    print(output)
```

### Advantages of This Approach:

- No compilation setup required
- Direct integration with Python for testing
- Can compare Python vs C++ implementations
- Full debugging capabilities
- Works on macOS, Linux, Windows

### Tips for Best Results:

- Always include error handling in run_cpp() calls
- Use std::cout for output (prints to Jupyter cells)
- Include necessary headers at the top
- Test compilation with small code snippets first